# Chapter 9 – Rethinking Digital Trust  
*If databases were birds, blockchain would be a duck*

This notebook explores how quantum computing changes the foundations of **digital trust**. We begin with a small blockchain called **DuckChain**, adapted from *Think Blockchain*, and use it to examine three core trust pillars: **digital identity**, **hashing**, and **consensus**.

We first build a classical version of DuckChain, then strengthen it with **ML-DSA**, a lattice-based digital signature algorithm designed for the post-quantum era. From there, we add **quantum-generated entropy** to support stronger key generation.

The notebook then introduces a simplified **zero-knowledge authorization** layer. Instead of recording which administrator approved a block, DuckChain records proof that an authorized participant approved it.

By the end, DuckChain becomes both **quantum-safe** and **quantum-enhanced**. The architecture remains familiar. The trust model evolves.

### Code 9-0: Classical DuckChain

This example recreates the original DuckChain from *Think Blockchain*, adapted from its JavaScript implementation into Python. The code demonstrates the three foundational trust pillars introduced in that book: **digital identity**, **hashing**, and **consensus**. Administrators approve transactions, hashes link blocks together, and consensus determines whether a block is accepted into the ledger.

**Note:** This example serves as the baseline for the chapter. The later code listings progressively enhance these same trust pillars with quantum-safe signatures, quantum-generated entropy, and privacy-preserving authorization proofs.

In [ ]:
# Code 9-0: DuckChain, a tiny teaching blockchain.
# Shows three core ingredients: hashing, signatures, and consensus.

import hashlib

##### Blockchain Ingredient 1: Hashing

# Create a SHA-256 fingerprint for any text input.
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Convert a block into the exact text we want to hash.
def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

# Hash a block without including its signatures.
def hash_block(block):
    return sha256(block_text(block))

##### Blockchain Ingredient 2: Signatures

# Create a toy signature for an admin approving a block hash.
def sign(admin, block_hash):
    return sha256(admin + block_hash)

# Record an admin vote by signing the proposed block hash.
def vote(admin, block):
    block["signatures"][admin] = sign(admin, block["hash"])

##### Blockchain Ingredient 3: Consensus

# Consensus means enough trusted admins signed this block.
def has_consensus(block, admins, required_signatures):
    votes = sum(
        block["signatures"].get(admin) == sign(admin, block["hash"])
        for admin in admins
    )
    return votes >= required_signatures

##### Block and chain utilities

# Create a block and hash its core contents.
def make_block(index, data, previous_hash):
    block = {
        "index": index,                  # position in the chain
        "data": data,                    # transaction or record
        "previous_hash": previous_hash,  # link to prior block
        "signatures": {},                # votes added after hashing
        "hash": ""
    }
    block["hash"] = hash_block(block)
    return block

# Verify that every block hashes correctly and links backward.
def validate_chain(chain, admins, required_signatures):
    for i in range(1, len(chain)):
        current = chain[i]
        previous = chain[i - 1]
        if current["hash"] != hash_block(current):
            return False
        if current["previous_hash"] != previous["hash"]:
            return False
        if not has_consensus(current, admins, required_signatures):
            return False
    return True

##### Demo: build a tiny DuckChain

# Step 1: Define the small network that can approve blocks.
admins = ["Huey", "Dewey", "Louie"]
required_signatures = 2

# Step 2: Start the chain with a genesis block.
duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

# Step 3: Propose a new block linked to the latest block.
transaction = "Donald pays Daisy 5 duck coins"
new_block = make_block(1, transaction, duckchain[-1]["hash"])

# Step 4: Collect enough signatures to reach consensus.
vote("Huey", new_block)
vote("Dewey", new_block)

# Step 5: Add the block only if consensus is reached.
if has_consensus(new_block, admins, required_signatures):
    duckchain.append(new_block)

# Step 6: Inspect and validate the resulting chain.
for block in duckchain:
    print(block)
print("DuckChain valid:",
      validate_chain(duckchain, admins, required_signatures))

### Code 9-1: Quantum-Safe DuckChain

This example upgrades the original DuckChain by replacing its identity layer with **ML-DSA**, a lattice-based digital signature algorithm designed to resist attacks from large-scale quantum computers. The code preserves the same three trust pillars introduced in *Think Blockchain*—digital identity, hashing, and consensus—but modernizes the identity layer using post-quantum cryptography. Administrators sign blocks using ML-DSA, consensus is established through signature verification, and the resulting ledger is validated to ensure its integrity.

**Note:** Before running this example, execute the notebook installation cell that installs the `dilithium-py` package. Later examples build on the functions defined here, so this code should be run before Codes 9-2, 9-3, and 9-4.

In [ ]:
# === Install dependency ===
!pip install dilithium-py --quiet

In [ ]:
# Code 9-1: Quantum-safe DuckChain
# Demonstrates digital identity, hashing, and consensus.

import hashlib
from dilithium_py.ml_dsa import ML_DSA_44

##### Trust Pillar 1: Digital Identity

# Create one ML-DSA keypair for each participant.
def make_admin_keys(admins):
    return {admin: ML_DSA_44.keygen() for admin in admins}

##### Trust Pillar 2: Hashing

# Create a SHA-256 fingerprint.
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Serialize the core fields of a block for hashing.
def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

# Hash a block without its signatures.
def hash_block(block):
    return sha256(block_text(block))

# Create a block linked to the previous block.
def make_block(index, data, previous_hash):
    block = {
        "index": index,                  # position in chain
        "data": data,                    # transaction payload
        "previous_hash": previous_hash,  # prior block link
        "signatures": {},                # approvals added after hashing
        "hash": ""
    }
    block["hash"] = hash_block(block)    # fingerprint of the block's contents
    return block

##### Trust Pillar 3: Consensus

# Sign a proposed block.
def vote(admin, block, admin_keys):
    public_key, secret_key = admin_keys[admin]
    block["signatures"][admin] = ML_DSA_44.sign(
        secret_key, block["hash"].encode("utf-8")
    )

# Verify one participant's signature.
def verify_vote(admin, block, admin_keys):
    if admin not in block["signatures"]:
        return False
    public_key, _ = admin_keys[admin]
    try:
        ML_DSA_44.verify(
            public_key,
            block["hash"].encode("utf-8"),
            block["signatures"][admin]
        )
        return True
    except Exception:
        return False

# Require enough valid signatures before accepting a block.
def has_consensus(block, admins, admin_keys, required_signatures):
    votes = sum(verify_vote(admin, block, admin_keys) for admin in admins)
    return votes >= required_signatures

##### Verification

# Verify hashes, chain links, and consensus for every block.
def validate_chain(chain, admins, admin_keys, required_signatures):
    for i in range(1, len(chain)):
        current, previous = chain[i], chain[i - 1]
        if current["hash"] != hash_block(current):
            return False
        if current["previous_hash"] != previous["hash"]:
            return False
        if not has_consensus(current, admins, admin_keys, required_signatures):
            return False
    return True

##### Step 1: Establish digital identities

admins = ["Huey", "Dewey", "Louie"]
admin_keys = make_admin_keys(admins)

##### Step 2: Create and hash records

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]
transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}
new_block = make_block(1, transaction, duckchain[-1]["hash"])

##### Step 3: Reach consensus

required_signatures = 2
vote("Huey", new_block, admin_keys)
vote("Dewey", new_block, admin_keys)
if has_consensus(new_block, admins, admin_keys, required_signatures):
    duckchain.append(new_block)

##### Verify the ledger

for block in duckchain:
    print({k: v for k, v in block.items() if k != "signatures"})
    print("signatures:", list(block["signatures"].keys()))

valid = validate_chain(duckchain, admins, admin_keys, required_signatures)
print("\nDuckChain valid:", valid)

### Code 9-2: Adding Quantum Randomness to DuckChain

This example introduces the first quantum enhancement to DuckChain by replacing classical entropy with **quantum-generated randomness**. A simple quantum circuit places qubits into superposition, measures them, and converts the resulting measurement outcomes into entropy for ML-DSA key generation. The blockchain architecture remains unchanged, but the digital identities now originate from quantum measurement rather than operating-system randomness.

**Note:** Before running this example, execute the notebook installation cells that install the `dilithium-py`, `qiskit`, and `qiskit-aer` packages. This code assumes that Code 9-1 has already been run, since it reuses the hashing, consensus, and ledger-validation functions defined there. Later examples build on the updated `make_admin_keys` function introduced in this listing.

In [ ]:
!pip -q install qiskit qiskit-aer pylatexenc

In [ ]:
# Code 9-2: Adding Quantum Randomness to DuckChain
# Assumes Code 9-1 has already been run.

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from dilithium_py.ml_dsa import ML_DSA_44

##### Quantum Enhancement 1: True Randomness

# Generate random bytes from quantum measurement outcomes.
def quantum_entropy(n_bytes):
    n_bits = n_bytes * 8                      # bytes → bits
    qc = QuantumCircuit(n_bits, n_bits)       # quantum and classical registers

    for i in range(n_bits):
        qc.h(i)                               # place each qubit into superposition

    qc.measure(range(n_bits), range(n_bits))  # measure all qubits

    result = AerSimulator().run(qc, shots=1).result()  # execute once
    bitstring = list(result.get_counts().keys())[0]    # measured bitstring

    # Convert quantum measurement outcomes into entropy bytes.
    return int(bitstring, 2).to_bytes(n_bytes, byteorder="big")

# Override Code 9-1 key generation with quantum-seeded key generation.
def make_admin_keys(admins):
    return {
        admin: ML_DSA_44.key_derive(
            quantum_entropy(32)  # randomness now comes from quantum measurement
        )
        for admin in admins
    }

##### Rebuild DuckChain using quantum-generated entropy

admins = ["Huey", "Dewey", "Louie"]
admin_keys = make_admin_keys(admins)

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}

new_block = make_block(1, transaction, duckchain[-1]["hash"])

required_signatures = 2
vote("Huey", new_block, admin_keys)
vote("Dewey", new_block, admin_keys)

if has_consensus(new_block, admins, admin_keys, required_signatures):
    duckchain.append(new_block)

valid = validate_chain(duckchain, admins, admin_keys, required_signatures)
print("DuckChain valid with quantum randomness:", valid)

### Code 9-3: Adding Zero-Knowledge Authorization to DuckChain

This example introduces the second quantum enhancement to DuckChain by replacing named signatures with **authorization proofs**. Rather than recording which administrator approved a block, the ledger records proof that an approved participant authorized it. The implementation uses a simplified hash-based construction to illustrate the core idea behind zero-knowledge systems: proving a claim without revealing unnecessary information. Consensus is preserved, but identity disclosure is reduced.

**Note:** This example assumes that Code 9-2 has already been run and reuses the hashing, ledger, and quantum-enhanced key-generation functions defined in the earlier listings. The authorization mechanism shown here is an educational approximation of a zero-knowledge proof and is intended to demonstrate the architectural pattern rather than provide production-grade cryptographic guarantees. The full quantum-safe, quantum-enhanced version of DuckChain is assembled in Code 9-4.

In [ ]:
# Code 9-3: Adding Zero-Knowledge Authorization to DuckChain
# Assumes Code 9-2 has already been run.

##### Quantum Enhancement 2: Zero-Knowledge Authorization

# Shared credential used only to simulate a ZKP.
# Real systems use cryptographic proof protocols.
AUTHORIZED_GROUP_CREDENTIAL = "DuckChain-Admins"

# Create a proof of authorization bound to this specific block.
def create_zk_proof(admin, block):
    return sha256(admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"])

# Verify that a proof belongs to some authorized admin for this block.
def verify_zk_proof(proof, block, admins):
    valid_proofs = {
        sha256(admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"])
        for admin in admins
    }
    return proof in valid_proofs  # proof matches an authorized participant

# Submit an authorization proof instead of a named signature.
def approve_block(admin, block):
    block["proofs"].append(create_zk_proof(admin, block))

# Count distinct valid authorization proofs.
def has_zk_consensus(block, admins, required_proofs):
    valid_proofs = {
        proof
        for proof in block["proofs"]
        if verify_zk_proof(proof, block, admins)
    }
    return len(valid_proofs) >= required_proofs

##### Rebuild DuckChain with authorization proofs

duckchain = [make_block(0, "Genesis Block Quack Quack!", "0")]

transaction = {
    "from": "Donald", "to": "Daisy",
    "amount": 5, "unit": "duck coins"
}

new_block = make_block(1, transaction, duckchain[-1]["hash"])
new_block["proofs"] = []  # replace named signatures with authorization proofs

approve_block("Huey", new_block)
approve_block("Dewey", new_block)

print("\nAuthorization proofs:")
for proof in new_block["proofs"]:
    print(proof[:16] + "...")  # truncate for readability

required_proofs = 2

if has_zk_consensus(new_block, admins, required_proofs):
    duckchain.append(new_block)

print("Proof count:", len(new_block["proofs"]))
print("Consensus reached:",
      has_zk_consensus(new_block, admins, required_proofs))

### Code 9-4: DuckChain, Quantum-Safe and Quantum-Enhanced

This example assembles the chapter's enhancements into a single reference implementation of DuckChain. The ledger combines **ML-DSA quantum-safe identities**, **quantum-generated entropy**, and **proof-based consensus** within the same blockchain. Transactions are hashed, linked into blocks, authorized through privacy-preserving proofs, and validated through the same three trust pillars introduced in *Think Blockchain*.

**Note:** This code is intended to be run after the earlier examples and serves as the chapter's consolidated DuckChain implementation. Before executing it, make sure the notebook installation cells have been run and that the required `dilithium-py`, `qiskit`, and `qiskit-aer` packages are available. The example brings together the concepts from Codes 9-1 through 9-3 and provides the foundation for the comparisons discussed later in the chapter.

In [ ]:
# Code 9-4: DuckChain, Quantum-Safe and Quantum-Enhanced
# A complete reference implementation.

import hashlib
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from dilithium_py.ml_dsa import ML_DSA_44

##### Quantum Enhancement 1: Generate Quantum Entropy

def quantum_entropy(n_bytes):
    n_bits = n_bytes * 8                    # bytes → bits
    qc = QuantumCircuit(n_bits, n_bits)     # quantum + classical registers

    for i in range(n_bits):
        qc.h(i)                             # place qubit into superposition

    qc.measure(range(n_bits), range(n_bits))  # measure all qubits

    result = AerSimulator().run(qc, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]

    return int(bitstring, 2).to_bytes(n_bytes, byteorder="big")


##### Trust Pillar 1: Quantum-Safe Digital Identity

def make_admin_keys(admins):
    return {
        admin: ML_DSA_44.key_derive(
            quantum_entropy(32)             # 256 bits of quantum entropy
        )
        for admin in admins
    }


##### Trust Pillar 2: Hashing

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def block_text(block):
    return f"{block['index']}|{block['data']}|{block['previous_hash']}"

def hash_block(block):
    return sha256(block_text(block))

def make_block(index, data, previous_hash):
    block = {
        "index": index,                    # block position
        "data": data,                      # transaction payload
        "previous_hash": previous_hash,    # link to previous block
        "proofs": [],                      # authorization proofs
        "hash": ""
    }
    block["hash"] = hash_block(block)
    return block


##### Quantum Enhancement 2: Zero-Knowledge Authorization

AUTHORIZED_GROUP_CREDENTIAL = "DuckChain-Admins"

def create_zk_proof(admin, block):
    return sha256(
        admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"]
    )

def verify_zk_proof(proof, block, admins):
    valid_proofs = {
        sha256(admin + AUTHORIZED_GROUP_CREDENTIAL + block["hash"])
        for admin in admins
    }
    return proof in valid_proofs              # authorized participant

def approve_block(admin, block):
    block["proofs"].append(
        create_zk_proof(admin, block)
    )


##### Trust Pillar 3: Proof-Based Consensus

def has_zk_consensus(block, admins, required_proofs):
    valid_proofs = {
        proof
        for proof in block["proofs"]
        if verify_zk_proof(proof, block, admins)
    }
    return len(valid_proofs) >= required_proofs


##### Verification

def validate_zk_chain(chain, admins, required_proofs):
    for i in range(1, len(chain)):
        current = chain[i]
        previous = chain[i - 1]

        if current["hash"] != hash_block(current):
            return False                      # block contents changed

        if current["previous_hash"] != previous["hash"]:
            return False                      # chain link broken

        if not has_zk_consensus(
            current, admins, required_proofs
        ):
            return False                      # consensus missing

    return True


##### Build DuckChain

admins = ["Huey", "Dewey", "Louie"]
admin_keys = make_admin_keys(admins)

duckchain = [
    make_block(0, "Genesis Block Quack Quack!", "0")
]

transaction = {
    "from": "Donald",
    "to": "Daisy",
    "amount": 5,
    "unit": "duck coins"
}

new_block = make_block(
    1,
    transaction,
    duckchain[-1]["hash"]
)

required_proofs = 2

approve_block("Huey", new_block)
approve_block("Dewey", new_block)

if has_zk_consensus(new_block, admins, required_proofs):
    duckchain.append(new_block)


##### Display Ledger Results

print("DuckChain valid:", validate_zk_chain(
    duckchain, admins, required_proofs
))

print("\nLatest block:")
print("index:", new_block["index"])
print("data:", new_block["data"])
print("previous_hash:", new_block["previous_hash"][:16] + "...")
print("hash:", new_block["hash"][:16] + "...")

print("\nAuthorization proofs stored in block:")
for proof in new_block["proofs"]:
    print(" ", proof[:16] + "...")

print("\nConsensus reached:",
      has_zk_consensus(new_block, admins, required_proofs))